In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Modules and Model Construction

The networks we have trained so far were small enough to write down layer by
layer. The networks ahead are not: deep ResNets chain more than a hundred
convolutional layers [@He.Zhang.Ren.ea.2016], and a GPT-style language
model stacks dozens of identical Transformer blocks
[@Radford.Wu.Child.ea.2019]. Such models are assembled from repeated
*blocks*, groups of layers treated as units of design. The abstraction that
makes blocks composable is the *module*.

A module is an object with three responsibilities: it owns *parameters*, it
owns *child modules*, and it implements a *forward computation* that maps
inputs to outputs. The definition is deliberately recursive. A fully connected
layer is a module (parameters, no children). A residual block is a module (no
direct parameters, a few child layers that own parameters). A hundred-layer
network is a module whose children are blocks whose children are layers. Most
models therefore have a tree-shaped module hierarchy, as sketched in
the figure. Reusing one child at several sites turns that tree into
an object graph, a case we meet when tying parameters in
that section. Almost everything
this chapter does to a model, listing its parameters
(that section), moving it to a GPU (that section),
saving it to disk (that section), is implemented as a walk over
that tree.

![Layers compose into blocks and blocks compose into models, giving the usual tree-shaped module hierarchy.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-tree.svg)

In [ ]:
from dataclasses import dataclass
from d2l import torch as d2l
import torch
from torch import nn
from torch.nn import functional as F

## The Module Abstraction

In PyTorch the module class is `nn.Module`. We have used one of its subclasses
all along: `nn.Sequential` builds a model from a chain of layers, here the
familiar MLP with a 256-unit ReLU hidden layer and a 10-unit output layer.

In [ ]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

X = torch.rand(2, 20)
net(X).shape

`Sequential` is not a special construct. It is itself a module whose forward
computation runs its children in order, and the children are the three modules
we passed to it, stored under names in a registry:

In [ ]:
net._modules

This registry is what the `nn.Module` machinery traverses: `net.parameters()`
collects parameters by walking `_modules` recursively, and the same walk
underlies device movement and serialization. A module the registry does not
contain might as well not exist, a fact we will exploit for a demonstration
shortly.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.out = nn.LazyLinear(10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [ ]:
net = MLP()
net(X).shape

Two details do the work here. First, `self.hidden = nn.LazyLinear(256)` is not
an ordinary attribute assignment: `nn.Module` intercepts `__setattr__`, sees
that the value is a module, and adds it to the registry we just inspected. That
is why both layers' parameters show up in `net.parameters()` with no further
ceremony. Second, we never wrote a backward method; automatic differentiation
derives gradients from whatever `forward` computes.

## Sequential and Friends: Containers

To see that there is no magic left in `nnx.Sequential`, we can write it
ourselves. Two ingredients suffice: register each child under a name, and loop
over the children in `forward`.

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)

    def forward(self, X):
        for module in self.children():
            X = module(X)
        return X

`add_module` writes a child into the registry under a string name (that is
where the `'0'`, `'1'`, `'2'` keys above came from), and `children()` iterates
the registry in insertion order. Our version is a drop-in replacement:

In [ ]:
net = MySequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net(X).shape

The registration step is easy to lose. The following module looks reasonable,
and its forward pass works, so nothing appears wrong:

In [ ]:
class PlainListMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)]

    def forward(self, X):
        for layer in self.layers:
            X = layer(X)
        return X

net = PlainListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

The model computes, yet it owns zero parameters. A plain Python list is not a
module, so the `__setattr__` interception ignores it and nothing inside it is
registered. The layers still hold perfectly good tensors, which is why
`forward` runs. But `net.parameters()` is empty: hand this model to an
optimizer and training proceeds without a single error while updating nothing;
`net.to(device)` leaves every layer behind on the CPU; `net.state_dict()`
checkpoints an empty model. Because no step raises an exception, this bug is
usually diagnosed by staring at a loss curve that refuses to move.

In [ ]:
class ModuleListMLP(PlainListMLP):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(self.layers)

net = ModuleListMLP()
net(X).shape, sum(p.numel() for p in net.parameters())

Same forward pass, 7946 registered parameters. The division of labor among the
containers is now clear. `nn.Sequential` registers its children and supplies
the run-them-in-order `forward`. `nn.ModuleList` registers a list of children
but supplies no `forward` at all; you keep writing the loop, which is exactly
what you want when the loop body is not a plain chain (blocks that take extra
arguments, or a skip connection around each block). `nn.ModuleDict` does the
same for children indexed by name. Transformer implementations conventionally
keep their stack of blocks in an `nn.ModuleList` and their named parts
(embedding, final normalization, output head) as attributes.

## Forward Is Just Python

`forward` is an ordinary Python method. Nothing restricts it to chaining
children: it can branch, loop, call any tensor function, and combine
intermediate results however it likes. The loop in `ModuleListMLP` already used
this freedom. Its most consequential one-line use is the *residual
connection*, the wiring idiom at the heart of ResNets and Transformers alike:

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(num_hiddens, num_hiddens), nn.ReLU(),
            nn.Linear(num_hiddens, num_hiddens))

    def forward(self, X):
        return X + self.body(X)

![The residual wiring `X + body(X)`: the input splits at a branch point into the body stack and an identity skip, and the two rejoin by addition before the block's output.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

`X + body(X)` is not a layer any framework provides. It is plain arithmetic
in the forward computation, and it changes what the block *is*: the block
computes a perturbation of the identity function rather than an arbitrary
transformation. If its body is $F$, its Jacobian is $I + J_F$, so
backpropagation receives an additive identity contribution along the skip
path. This contribution can still cancel against $J_F$; it is a direct path,
not a guarantee that every gradient is preserved
[@He.Zhang.Ren.ea.2016]. the figure diagrams
exactly this wiring, and that section develops both points when we
build ResNet; for now we only need the mechanics. One mechanical consequence
is visible already: the addition forces the input and output shapes to
agree, so a residual block has a single width that is part of its identity.

That is why we gave `body` explicit `nn.Linear` layers rather than lazy ones.

In [ ]:
block = ResidualBlock(24)
block(torch.randn(2, 24)).shape

`forward` may also use state that is neither an input nor a parameter. Suppose
we want to damp each block's contribution by a fixed factor:

In [ ]:
class ScaledResidual(ResidualBlock):
    def __init__(self, num_hiddens, alpha=0.5):
        super().__init__(num_hiddens)
        self.alpha = torch.tensor(alpha)  # Fixed by design, never trained

    def forward(self, X):
        return X + self.alpha * self.body(X)

block = ScaledResidual(24)
'alpha' in block.state_dict(), list(block.state_dict())[:2]

`alpha` enters the computation, but it is not a parameter: it never appears in
`named_parameters()`, so the optimizer never touches it. That much we wanted.
Storing it as a plain attribute has a cost we did not want, though: as the
output shows, it is missing from `state_dict()` as well, so it will not be
saved with the model, and `.to(device)` will not move it. Some state is not a
parameter but must still travel with the model; the registered home for such
state is a *buffer*, introduced in that section.

## Lazy Initialization: Shapes from Data

We have been doing something odd since our first MLP without commenting on it:
`nn.LazyLinear(256)` names only the layer's *output* width. Its weight matrix
has shape `(256, in_features)`, and we never said what `in_features` is.
The layer cannot know it at construction time, since it depends on the data it
will receive. So it does not allocate parameters at construction time at all:

In [ ]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))
net[0].weight

The weight is a placeholder. The first time data flows through, the layer
reads the input width from the batch, allocates and initializes a real weight,
and replaces itself with a plain `nn.Linear`. Its output width then fixes the
input of the next lazy layer, and shapes cascade through the whole model:

In [ ]:
net(X)
net[0].weight.shape

Lazy layers can remove shape arithmetic from model definitions. In a
convolutional network the flattened feature-map size depends on the input
resolution and upstream strides. We usually avoid that fragile arithmetic by
using global or adaptive pooling before the first linear layer. When a width
is part of the architecture, we name it in the model config and pass it to
adjacent layers — code that stays explicit and compact without hiding
parameter creation behind a sample batch, whether or not the framework
offers a lazy mode.

Before the first forward pass, lazy layers contain registered
`UninitializedParameter` placeholders rather than shaped arrays. An optimizer
can be constructed over those placeholders, but reading shapes, counting
elements, or applying a shape-dependent initializer requires a *dry run* on a
representative batch. A related subtlety:
the random initialization now happens at first call rather than at
construction, so any random numbers your program draws in between shift the
generator's state, and a fixed seed can yield different weights than the
explicitly shaped version of the same model (that section returns to
seeding). NNX has no lazy mode: its linear layers require both widths and
create parameters in the constructor. PyTorch's lazy layers instead let the
first real batch supply the missing width.

In [ ]:
@d2l.add_to_class(d2l.Module)
def apply_init(self, inputs, init=None):
    self.forward(*inputs)
    if init is not None:
        self.net.apply(init)

`nn.Module.apply(fn)` calls `fn` on every module in the tree, children first.
It is the standard way to push a policy across an arbitrary model, one more
operation that is a tree walk, and from that section on the idiom
`model.apply_init([X], init)` opens most of our training scripts. A small
demonstration:

In [ ]:
class TinyMLP(d2l.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(),
                                 nn.LazyLinear(10))

def init_xavier(module):
    if type(module) == nn.Linear:
        nn.init.xavier_uniform_(module.weight)

model = TinyMLP()
model.apply_init([X], init_xavier)
model.net[0].weight.shape

The dry run inside `apply_init` turned every lazy layer into a real one, after
which `init_xavier` could match on `nn.Linear` and rewrite its weights. Which
initializer to apply, and why Xavier's variance rule
(that section) is a sensible default, is the subject of
that section.

## Building from a Config

So far every width in this section was a literal typed into a constructor.
Real model code does not work that way. The architecture choices that vary
across experiments, such as depth, width, and output size, must be logged with
results and stored with checkpoints so that a saved model can be rebuilt. The
topology remains in code; a small configuration object records its variable
choices:

In [ ]:
@dataclass
class MLPConfig:
    d_in: int = 784
    d_hidden: int = 256
    num_blocks: int = 4
    d_out: int = 10

def build(cfg: MLPConfig) -> nn.Module:
    blocks = [ResidualBlock(cfg.d_hidden) for _ in range(cfg.num_blocks)]
    return nn.Sequential(nn.Linear(cfg.d_in, cfg.d_hidden),
                         *blocks, nn.Linear(cfg.d_hidden, cfg.d_out))

One config produces one architecture: an input projection into the hidden
width, `num_blocks` identical residual blocks, and an output projection. The
list comprehension producing `blocks` is a plain Python list, which is safe
here for the reason that section taught:
unpacking it into `nn.Sequential` is what registers each block. Every layer
gets explicit widths, and this is where explicit shapes beat lazy ones: the
config already knows every width, so explicit construction yields a fully
materialized model with no dry run needed. Printing the model displays the
module tree:

In [ ]:
net = build(MLPConfig())
net

In [ ]:
net(torch.rand(2, 784)).shape

The variable architecture choices are now data. Rescaling the model is a change to two fields, not
an edit to model code:

In [ ]:
for cfg in (MLPConfig(), MLPConfig(d_hidden=512, num_blocks=8)):
    n = sum(p.numel() for p in build(cfg).parameters())
    print(f'd_hidden={cfg.d_hidden}, num_blocks={cfg.num_blocks}: '
          f'{n:,} parameters')

Because `build` is deterministic in `cfg`, the config is all you need to
reconstruct the module tree later; that section saves it
alongside the weights so that loading a checkpoint starts by rebuilding the
exact same model. A config of widths and depths feeding a loop that stacks
identical residual blocks is a common construction pattern in the Transformer
implementations later in the book.

## Summary

A module owns parameters, child modules, and a `forward` method. Layers,
blocks, and whole models are the same kind of object, so the usual hierarchy
is tree-shaped, while shared children introduce aliases. Parameter collection,
device movement, and serialization traverse that object graph. Children are
discovered through registration: attribute
assignment and the containers `nn.Sequential`, `nn.ModuleList`, and
`nn.ModuleDict` register children, while a plain Python list hides them,
yielding a model that runs but trains nothing. `forward` is ordinary Python;
a residual connection is one line in it. Lazy layers declare output widths and
infer input widths on the first forward pass, so initialization and inspection
follow a dry run (`apply_init`). Configs turn architecture into data: a
`dataclass` of widths and depths plus a `build` function that stacks blocks.

## Exercises

1. Take `PlainListMLP` and catalog everything that breaks besides the empty
   parameter list. Check `net.state_dict()`, the effect of
   `net.to(torch.float64)` on the hidden layers' dtypes, and whether
   `net.eval()` reaches the children. Explain how each failure follows from
   the same missing registration.
1. Implement a `ParallelBlock` that takes two child modules `net1` and `net2`,
   runs both on the same input, and concatenates their outputs along the last
   dimension. What must be true of the two children's outputs for the
   concatenation to be valid?
1. Extend `MLPConfig` with an activation switch (for example,
   `act: str = 'relu'`) and make `build` honor it. Which decisions belong in a
   config and which belong in code? Where would you put a choice between
   `ResidualBlock` and a plain feed-forward block?
1. `ResidualBlock` requires its input and output widths to agree. Suppose you
   want a block whose output is wider than its input. Give two standard fixes
   and the cost of each. (that section uses one of them in ResNet.)